# **import stable xl and required support**

In [ ]:
from huggingface_hub import login
login("enter your api-key")

In [ ]:
!pip install diffusers transformers accelerate DeepCache --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 17.4 MB/s eta 0:00:00


# **Linear**

In [ ]:
# ============================================================
# Linear Schedule DeepCache vs Official DeepCache
# ============================================================

import os
import csv
import time
import math
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from typing import List, Set

import numpy as np
import torch
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Device: {DEVICE}, torch: {torch.__version__}")

# ============================================================
# 1. loading model
# ============================================================

from diffusers import StableDiffusionXLPipeline, DDIMScheduler
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=DTYPE,
    variant="fp16",
    use_safetensors=True,
).to(DEVICE)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

# ============================================================
# 2. compute PSNR , Image similarity, CLIP score
# ============================================================

from transformers import CLIPProcessor, CLIPModel

print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)



def compute_psnr(img_ref, img_gen):
    a = np.array(img_ref.convert("RGB")).astype(np.float32)
    b = np.array(img_gen.convert("RGB")).astype(np.float32)
    mse = np.mean((a - b) ** 2)
    if mse == 0:
        return 100.0
    return float(20 * np.log10(255.0 / (np.sqrt(mse) + 1e-8)))


@torch.no_grad()
def compute_clip_image_similarity(img_ref, img_gen):
    inp_ref = clip_processor(images=img_ref.convert("RGB"), return_tensors="pt").to(DEVICE)
    inp_gen = clip_processor(images=img_gen.convert("RGB"), return_tensors="pt").to(DEVICE)
    f_ref = clip_model.get_image_features(**inp_ref)
    f_gen = clip_model.get_image_features(**inp_gen)
    if not isinstance(f_ref, torch.Tensor):
        f_ref = f_ref.last_hidden_state[:, 0]
    if not isinstance(f_gen, torch.Tensor):
        f_gen = f_gen.last_hidden_state[:, 0]
    f_ref = f_ref / f_ref.norm(dim=-1, keepdim=True)
    f_gen = f_gen / f_gen.norm(dim=-1, keepdim=True)
    return float((f_ref @ f_gen.T).item())


@torch.no_grad()
def compute_clip_score(image, text):
    inputs = clip_processor(text=[text], images=image.convert("RGB"),
                            return_tensors="pt", padding=True,
                            truncation=True, max_length=77)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    out = clip_model(**inputs)
    ie = out.image_embeds
    te = out.text_embeds
    if ie is None:
        ie = out.vision_model_output.last_hidden_state[:, 0]
        ie = clip_model.visual_projection(ie)
    if te is None:
        te = out.text_model_output.last_hidden_state[:, 0]
        te = clip_model.text_projection(te)
    ie = ie / ie.norm(dim=-1, keepdim=True)
    te = te / te.norm(dim=-1, keepdim=True)
    return float((ie @ te.T).item())


# ============================================================
# 3. Linearly increasing interval
# ============================================================

def compute_deepcache_budget(total_steps: int, cache_interval: int) -> int:
    # Calculate the number of Full UNet iterations of the official DeepCache at a given interval
    return total_steps // cache_interval + (1 if total_steps % cache_interval else 0)

def make_linear_schedule(T: int, K: int, min_gap: int = 2) -> List[int]:

    if K <= 1:
        return [0]
    if K >= T:
        return list(range(T))

    n = K - 1  # Number of intervals between K full steps
    total_span = T - 1 # Total distance from step 0 to step T-1


    d_min = max(min_gap, 1.0) # Determine d_min (the smallest interval, at the beginning)

    # # Feasibility check
    if n * d_min > total_span:
        # K is too large for this min_gap, so we shrink d_min to make uniform spacing
        d_min = total_span / n

    # calculate the linear growth rate of intervals Δ
    if n > 1:
        delta_inc = (total_span - n * d_min) / (n * (n - 1) / 2)
    else:
        delta_inc = 0

    # if delta_inc < 0, d_min is already too large. Fall back to uniform spacing
    if delta_inc < 0:
        delta_inc = 0
        d_min = total_span / n

    # Generate the interval sequence
    intervals = [d_min + i * delta_inc for i in range(n)]

    # Convert intervals to cumulative positions (floating point)
    positions = [0.0]
    for gap in intervals:
        positions.append(positions[-1] + gap)

    # Snap floating-point positions to integer step indices
    steps = [0]
    for j in range(1, len(positions)):
        s = int(round(positions[j]))
        s = max(s, steps[-1] + min_gap)  # Enforce minimum gap constraint
        s = min(s, T - 1)         # Clamp to valid range [0, T-1]
        if s not in steps:
            steps.append(s)

    # Backfill if we ended up with fewer than K steps
    while len(steps) < K:
        for s in range(T - 1, -1, -1):
            if s not in steps:
                # Check min_gap against ALL existing full steps
                ok = True
                for existing in steps:
                    if abs(s - existing) < min_gap:
                        ok = False
                        break
                if ok:
                    steps.append(s)
                    steps.sort()
                    break
        else:
            # find any unused step index to fill the budget
            for s in range(T - 1, -1, -1):
                if s not in steps:
                    steps.append(s)
                    steps.sort()
                    break

    steps = sorted(steps[:K])
    return steps

# Visualization of the process
def visualize_schedule(name: str, steps: List[int], T: int):
    step_set = set(steps)
    trace = ''.join('F' if s in step_set else '·' for s in range(T))
    gaps = [steps[i+1] - steps[i] for i in range(len(steps)-1)]
    print(f"  {name:<16} K={len(steps):>2}  {trace}")
    print(f"  {'':16}       gaps: {gaps}")


# ============================================================
# 4. Schedule-Based Cache Helper
# ============================================================

@dataclass
class ScheduleCacheConfig:
    total_steps: int = 50
    cache_branch_id: int = 0


class ScheduleCacheHelper:
    """ Given a pre-computed list of full step indices, this helper wraps
    the UNet so that:
      - On FULL steps: every module computes normally, outputs are cached
      - On CACHE steps: shallow modules compute normally, deep modules
        return cached outputs from the last full step """

    def __init__(self, pipe, config: ScheduleCacheConfig, full_step_list: List[int]):
        self.pipe = pipe
        self.config = config
        self.full_step_set = set(full_step_list)
        self.full_step_list = sorted(full_step_list)
        self.function_dict = {}
        self.cached_output = {}
        self.step_log = []
        self.steps_done = 0
        self._is_full_step = True

        self.cache_block_id = config.cache_branch_id // 3
        self.cache_layer_id = config.cache_branch_id % 3

    def _is_skip_block(self, block_i, layer_i, blocktype="down"):
        # replicates the official DeepCache spatial caching
        if block_i > self.cache_block_id or blocktype == 'mid':
            return True
        if block_i < self.cache_block_id:
            return False
        if blocktype == 'down':
            return layer_i >= self.cache_layer_id
        else:
            return layer_i > self.cache_layer_id

    def _wrap_unet_forward(self):
        # Wrap the top-level UNet forward function
        self.function_dict['unet_forward'] = self.pipe.unet.forward
        helper = self

        def wrapped_forward(*args, **kwargs):
            is_full = helper.steps_done in helper.full_step_set
            helper._is_full_step = is_full
            result = helper.function_dict['unet_forward'](*args, **kwargs)
            helper.step_log.append({
                'step': helper.steps_done,
                'is_full': is_full,
            })
            helper.steps_done += 1
            return result

        self.pipe.unet.forward = wrapped_forward

    def _wrap_block_forward(self, block, block_name, block_i, layer_i, blocktype="down"):
        # Wrap a single sub-module's forward function
        key = (blocktype, block_name, block_i, layer_i)
        self.function_dict[key] = block.forward
        helper = self

        def wrapped_forward(*args, **kwargs):
            if helper._is_full_step:
                result = helper.function_dict[key](*args, **kwargs)
                helper.cached_output[key] = result
                return result
            else:
                if helper._is_skip_block(block_i, layer_i, blocktype) and key in helper.cached_output:
                    return helper.cached_output[key]
                result = helper.function_dict[key](*args, **kwargs)
                helper.cached_output[key] = result
                return result

        block.forward = wrapped_forward


    def _wrap_modules(self):
        # Walk through the entire UNet and wrap every relevant sub-module.
        self._wrap_unet_forward()

        # Down blocks: natural order, indices unchanged
        for bi, blk in enumerate(self.pipe.unet.down_blocks):
            for li, a in enumerate(getattr(blk, "attentions", [])):
               self._wrap_block_forward(a, "attentions", bi, li, "down")
            for li, r in enumerate(getattr(blk, "resnets", [])):
               self._wrap_block_forward(r, "resnet", bi, li, "down")
            for d in (getattr(blk, "downsamplers", []) if blk.downsamplers else []):
               self._wrap_block_forward(d, "downsampler", bi, len(getattr(blk, "resnets", [])), "down")

        # Mid block: always the deepest, always cached on cache steps
        self._wrap_block_forward(self.pipe.unet.mid_block, "mid_block", 0, 0, "mid")

        # Up blocks: both block and layer indices reversed
        bn = len(self.pipe.unet.up_blocks)
        for bi, blk in enumerate(self.pipe.unet.up_blocks):
            mapped_bi = bn - bi - 1  # Block index reversed
            ln = len(getattr(blk, "resnets", []))
            for li, a in enumerate(getattr(blk, "attentions", [])):
               self._wrap_block_forward(a, "attentions", mapped_bi, ln-li-1, "up")
            for li, r in enumerate(getattr(blk, "resnets", [])):
               self._wrap_block_forward(r, "resnet", mapped_bi, ln-li-1, "up")
            for u in (getattr(blk, "upsamplers", []) if blk.upsamplers else []):
               self._wrap_block_forward(u, "upsampler", mapped_bi, 0, "up")

    def _unwrap_modules(self):
        # Restore all original forward functions
        self.pipe.unet.forward = self.function_dict['unet_forward']

        # Down blocks: same natural-order keys
        for bi, blk in enumerate(self.pipe.unet.down_blocks):
           for li, a in enumerate(getattr(blk, "attentions", [])):
              a.forward = self.function_dict[("down", "attentions", bi, li)]
           for li, r in enumerate(getattr(blk, "resnets", [])):
              r.forward = self.function_dict[("down", "resnet", bi, li)]
           for d in (getattr(blk, "downsamplers", []) if blk.downsamplers else []):
              d.forward = self.function_dict[("down", "downsampler", bi, len(getattr(blk, "resnets", [])))]
        # Mid block
        self.pipe.unet.mid_block.forward = self.function_dict[("mid", "mid_block", 0, 0)]
        # Up blocks: same reversed-index keys as _wrap_modules
        bn = len(self.pipe.unet.up_blocks)
        for bi, blk in enumerate(self.pipe.unet.up_blocks):
           mapped_bi = bn - bi - 1
           ln = len(getattr(blk, "resnets", []))
           for li, a in enumerate(getattr(blk, "attentions", [])):
              a.forward = self.function_dict[("up", "attentions", mapped_bi, ln-li-1)]
           for li, r in enumerate(getattr(blk, "resnets", [])):
              r.forward = self.function_dict[("up", "resnet", mapped_bi, ln-li-1)]
           for u in (getattr(blk, "upsamplers", []) if blk.upsamplers else []):
              u.forward = self.function_dict[("up", "upsampler", mapped_bi, 0)]



    def enable(self):
        self._reset()
        self._wrap_modules()

    def disable(self):
        self._unwrap_modules()
        self._reset()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _reset(self):
        self.function_dict = {}
        self.cached_output = {}
        self.step_log = []
        self.steps_done = 0
        self._is_full_step = True

    @contextmanager
    def enabled(self):
        self.enable()
        try:
            yield self
        finally:
            self.disable()

    def get_stats(self):
        if not self.step_log:
            return {}
        full_steps = sum(1 for s in self.step_log if s['is_full'])
        cache_steps = sum(1 for s in self.step_log if not s['is_full'])
        return {
            'total_steps': len(self.step_log),
            'full_steps': full_steps,
            'cache_steps': cache_steps,
            'full_trace': ['F' if s['is_full'] else 's' for s in self.step_log],
        }


# ============================================================
# 5. Generation functions
# ============================================================

def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def gen_baseline(prompt, steps=50, seed=42):
    # Generate an image with standard DDIM sampling (no caching)
    set_seed(seed)
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    return img, time.time() - t0


def gen_deepcache(prompt, steps=50, seed=42, cache_interval=3, cache_branch_id=0):
    # Generate an image using the official DeepCache library
    from DeepCache import DeepCacheSDHelper
    set_seed(seed)
    h = DeepCacheSDHelper(pipe=pipe)
    h.set_params(cache_interval=cache_interval, cache_branch_id=cache_branch_id)
    h.enable()
    t0 = time.time()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    elapsed = time.time() - t0
    h.disable()
    return img, elapsed


def gen_linear(prompt, steps, seed, full_step_list, branch_id=0):
    # Generate an image using our linear schedule caching
    set_seed(seed)
    cfg = ScheduleCacheConfig(total_steps=steps, cache_branch_id=branch_id)
    h = ScheduleCacheHelper(pipe, cfg, full_step_list)
    with h.enabled():
        t0 = time.time()
        with torch.inference_mode():
            img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
        elapsed = time.time() - t0
        stats = h.get_stats()
    return img, elapsed, stats


# ============================================================
# 6. Run experiment
# ============================================================

CSV_PATH = "prompts_LITTLE4.csv"
PROMPT_COLUMN = "prompt"
NUM_STEPS = 50
SEED = 42
DC_INTERVAL = 5
DC_BRANCH = 3
SAVE_DIR = "results"

# Compute the full UNet budget from DeepCache settings
FULL_BUDGET = compute_deepcache_budget(NUM_STEPS, DC_INTERVAL)

# Generate linear schedule
linear_schedule = make_linear_schedule(NUM_STEPS, FULL_BUDGET)

print(f"DeepCache: interval={DC_INTERVAL}, branch={DC_BRANCH}, Full K={FULL_BUDGET}")
print(f"\nSchedule comparison (T={NUM_STEPS}, K={FULL_BUDGET}):")
print(f"{'─'*70}")
# DeepCache's uniform positions
dc_positions = list(range(0, NUM_STEPS, DC_INTERVAL))
visualize_schedule("DeepCache", dc_positions, NUM_STEPS)
visualize_schedule("Linear", linear_schedule, NUM_STEPS)
print(f"{'─'*70}")
assert len(linear_schedule) == FULL_BUDGET, \
    f"Linear schedule K={len(linear_schedule)} != budget {FULL_BUDGET}"
print(f" Both methods use exactly {FULL_BUDGET}, Fair comparison\n")

# Load prompts from CSV
prompts = []
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        t = row.get(PROMPT_COLUMN, "").strip()
        if t:
            prompts.append(t)
print(f"{len(prompts)} prompts loaded from {CSV_PATH}\n")


os.makedirs(SAVE_DIR, exist_ok=True)
all_results = []

out_csv = os.path.join(SAVE_DIR, "results.csv")
out_fields = [
    "run", "prompt", "budget_K", "branch_id",
    "time_bl", "time_dc", "time_linear",
    "speedup_dc", "speedup_linear",
    "psnr_dc", "psnr_linear",
    "sim_dc", "sim_linear",
    "clip_bl", "clip_dc", "clip_linear",
    "dc_full", "linear_full",
    "trace_dc", "trace_linear",
]

with open(out_csv, "w", newline="", encoding="utf-8") as cf:
    cw = csv.DictWriter(cf, fieldnames=out_fields)
    cw.writeheader()

    for i, prompt in enumerate(prompts):
        print(f"\n{'='*60}\n  Run {i+1}/{len(prompts)}  [K={FULL_BUDGET}, branch={DC_BRANCH}]\n{'='*60}")
        print(f'  Prompt: "{prompt[:80]}{"..." if len(prompt)>80 else ""}"')

        # 1. Baseline
        print("  [1/3] Baseline ...")
        img_bl, t_bl = gen_baseline(prompt, NUM_STEPS, SEED)
        clip_bl = compute_clip_score(img_bl, prompt)
        print(f"        {t_bl:.2f}s  CLIP={clip_bl:.4f}")

        # 2. Official DeepCache
        print(f"  [2/3] DeepCache (official, interval={DC_INTERVAL}, branch={DC_BRANCH}) ...")
        img_dc, t_dc = gen_deepcache(prompt, NUM_STEPS, SEED, DC_INTERVAL, DC_BRANCH)
        sp_dc = t_bl / max(t_dc, 1e-6)
        psnr_dc = compute_psnr(img_bl, img_dc)
        sim_dc = compute_clip_image_similarity(img_bl, img_dc)
        clip_dc = compute_clip_score(img_dc, prompt)
        print(f"        {t_dc:.2f}s ({sp_dc:.2f}x)  PSNR={psnr_dc:.1f}  CLIP={clip_dc:.4f}")

        # 3. Linear Schedule
        print(f"  [3/3] Linear Schedule (K={FULL_BUDGET}, branch={DC_BRANCH}) ...")
        img_li, t_li, stats_li = gen_linear(
            prompt, NUM_STEPS, SEED, linear_schedule, branch_id=DC_BRANCH)
        sp_li = t_bl / max(t_li, 1e-6)
        psnr_li = compute_psnr(img_bl, img_li)
        sim_li = compute_clip_image_similarity(img_bl, img_li)
        clip_li = compute_clip_score(img_li, prompt)
        trace_li = ''.join(stats_li['full_trace'])
        print(f"        {t_li:.2f}s ({sp_li:.2f}x)  PSNR={psnr_li:.1f}  CLIP={clip_li:.4f}")
        print(f"        Full={stats_li['full_steps']}  Trace: {trace_li}")


        print(f"\n  {'Method':<20} {'Time':>6} {'Spdup':>6} {'PSNR':>7} {'ImgSim':>7} {'CLIP':>7} {'Full':>4}")
        print(f"  {'-'*62}")
        print(f"  {'Baseline':<20} {t_bl:>6.2f} {'1.00x':>6} {'--':>7} {'--':>7} {clip_bl:>7.4f} {NUM_STEPS:>4}")
        print(f"  {'DeepCache(official)':<20} {t_dc:>6.2f} {sp_dc:>5.2f}x {psnr_dc:>7.1f} {sim_dc:>7.4f} {clip_dc:>7.4f} {FULL_BUDGET:>4}")
        print(f"  {'Linear(ours)':<20} {t_li:>6.2f} {sp_li:>5.2f}x {psnr_li:>7.1f} {sim_li:>7.4f} {clip_li:>7.4f} {stats_li['full_steps']:>4}")

        W, H = img_bl.size
        comp = Image.new("RGB", (W * 3 + 20, H + 30), "white")
        draw = ImageDraw.Draw(comp)
        for j, (img, title) in enumerate(zip(
            [img_bl, img_dc, img_li],
            ["Baseline",
             f"DeepCache(i={DC_INTERVAL})",
             f"Linear(K={FULL_BUDGET})"]
        )):
            x = j * (W + 10)
            comp.paste(img, (x, 25))
            draw.text((x + W // 2, 5), title, fill="black", anchor="mt")
        comp_path = os.path.join(SAVE_DIR, f"cmp_{i+1}_s{SEED}.png")
        comp.save(comp_path)
        print(f"\n  Saved -> {comp_path}")
        try:
            from IPython.display import display
            display(comp)
        except:
            pass
        r = {
            'times': {'baseline': t_bl, 'deepcache': t_dc, 'linear': t_li},
            'speedups': {'deepcache': sp_dc, 'linear': sp_li},
            'psnr': {'deepcache': psnr_dc, 'linear': psnr_li},
            'sim': {'deepcache': sim_dc, 'linear': sim_li},
            'clip': {'baseline': clip_bl, 'deepcache': clip_dc, 'linear': clip_li},
        }
        all_results.append(r)

        cw.writerow({
            "run": i + 1, "prompt": prompt[:100],
            "budget_K": FULL_BUDGET, "branch_id": DC_BRANCH,
            "time_bl": f"{t_bl:.3f}", "time_dc": f"{t_dc:.3f}", "time_linear": f"{t_li:.3f}",
            "speedup_dc": f"{sp_dc:.3f}", "speedup_linear": f"{sp_li:.3f}",
            "psnr_dc": f"{psnr_dc:.2f}", "psnr_linear": f"{psnr_li:.2f}",
            "sim_dc": f"{sim_dc:.4f}", "sim_linear": f"{sim_li:.4f}",
            "clip_bl": f"{clip_bl:.4f}", "clip_dc": f"{clip_dc:.4f}", "clip_linear": f"{clip_li:.4f}",
            "dc_full": FULL_BUDGET, "linear_full": stats_li['full_steps'],
            "trace_dc": ''.join('F' if s in set(dc_positions) else 's' for s in range(NUM_STEPS)),
            "trace_linear": trace_li,
        })
        cf.flush()


# ============================================================
# 7. Summary
# ============================================================

print(f"\n{'='*70}")
print(f"  SUMMARY ({len(all_results)} runs)")
print(f"  Budget K={FULL_BUDGET}, branch_id={DC_BRANCH}")
print(f"{'='*70}")

sp_dc = np.mean([r['speedups']['deepcache'] for r in all_results])
sp_li = np.mean([r['speedups']['linear'] for r in all_results])
p_dc  = np.mean([r['psnr']['deepcache'] for r in all_results])
p_li  = np.mean([r['psnr']['linear'] for r in all_results])
s_dc  = np.mean([r['sim']['deepcache'] for r in all_results])
s_li  = np.mean([r['sim']['linear'] for r in all_results])
c_bl  = np.mean([r['clip']['baseline'] for r in all_results])
c_dc  = np.mean([r['clip']['deepcache'] for r in all_results])
c_li  = np.mean([r['clip']['linear'] for r in all_results])

print(f"  {'Method':<24} {'Speedup':>8} {'PSNR(dB)':>10} {'ImgSim':>8} {'CLIP':>8} {'Full':>5}")
print(f"  {'-'*65}")
print(f"  {'DDIM Baseline':<24} {'1.00x':>8} {'--':>10} {'--':>8} {c_bl:>8.4f} {NUM_STEPS:>5}")
print(f"  {'DeepCache(official)':<24} {sp_dc:>7.2f}x {p_dc:>10.1f} {s_dc:>8.4f} {c_dc:>8.4f} {FULL_BUDGET:>5}")
print(f"  {'Linear(ours)':<24} {sp_li:>7.2f}x {p_li:>10.1f} {s_li:>8.4f} {c_li:>8.4f} {FULL_BUDGET:>5}")
print()
print(f"  Full UNet = {FULL_BUDGET}， branch_id = {DC_BRANCH}")

print(f"{'='*70}")
print(f"  CSV -> {out_csv}")
print("Done!")

# **HYPOTHESIE TESING**

In [ ]:
# ============================================================
# HYPOTHESIE TESING
# ============================================================

import numpy as np
from scipy import stats

# Significance level: probability threshold for rejecting H₀
alpha = 0.05


METHOD_KEY = 'linear'
METHOD_NAME = 'Linear Schedule'

# Collect paired measurements for each metric
metrics = {
    'PSNR (dB)': (
        [r['psnr'][METHOD_KEY] for r in all_results],
        [r['psnr']['deepcache'] for r in all_results],
    ),
    'CLIP Image Similarity': (
        [r['sim'][METHOD_KEY] for r in all_results],
        [r['sim']['deepcache'] for r in all_results],
    ),
    'CLIP Score': (
        [r['clip'][METHOD_KEY] for r in all_results],
        [r['clip']['deepcache'] for r in all_results],
    ),
    'Speedup': (
        [r['speedups'][METHOD_KEY] for r in all_results],
        [r['speedups']['deepcache'] for r in all_results],
    ),
}


print(f"{'=' * 70}")
print(f"  HYPOTHESIE TESING: {METHOD_NAME} vs DeepCache")
print(f"  H₀: μ(ours - deepcache) ≤ 0  ({METHOD_NAME} is NOT better than DeepCache)")
print(f"  H₁: μ(ours - deepcache) > 0  ({METHOD_NAME} is better than DeepCache)")
print(f"  Method: Paired t-test (one-tailed,α = {alpha})")
print(f"  Sample size:{len(all_results)} prompts")
print(f"{'=' * 70}")

summary = []

for metric_name, (ours, dc) in metrics.items():

    # ── Step 1: Compute paired differences ──
    diffs = np.array(ours) - np.array(dc)
    n = len(diffs)

    # ── Step 2: Descriptive statistics of the differences ──
    mean_d = np.mean(diffs)
    std_d = np.std(diffs, ddof=1)
    se = std_d / np.sqrt(n)
    df = n - 1

    # ── Step 3: Paired t-test ──
    t_stat = mean_d / se if se > 0 else 0.0
    p_val = stats.t.sf(t_stat, df=df)

    # ── Step 4: Effect size (Cohen's d) ──
    cohens_d = mean_d / std_d if std_d > 0 else 0.0

    # ── Step 5: One-sided confidence interval ──
    ci_lower = mean_d - stats.t.ppf(1 - alpha, df=df) * se

    # ── Step 6: Normality check (Shapiro-Wilk test) ──
    if n >= 8:
        _, shapiro_p = stats.shapiro(diffs)
        norm_ok = shapiro_p > 0.05
    else:
        shapiro_p = float('nan')
        norm_ok = None

    print(f"\n  [{metric_name}]")
    print(f"  {'─' * 55}")
    p_str = f"{p_val:.2e}" if p_val < 1e-6 else f"{p_val:.6f}"
    print(f"  n={n}  mean(diff)={mean_d:+.4f}  std={std_d:.4f}  SE={se:.4f}")
    print(f"  t={t_stat:.4f}  df={df}  p={p_str}  Cohen's d={cohens_d:.4f}")
    print(f"  {1-alpha:.0%} confidence lower bound: {ci_lower:+.4f}")

    if norm_ok is not None:
        print(f"  Shapiro-Wilk: p={shapiro_p:.4f} → {'Normal' if norm_ok else 'Non-normal'}")

    # ── Step 7: Interpret significance level ──
    if p_val < 0.001:
        conclusion = f" Highly significant (p={p_str}), reject H₀"
        sig = True
    elif p_val < 0.01:
        conclusion = f" Very significant (p={p_str}), reject H₀"
        sig = True
    elif p_val < 0.05:
        conclusion = f" Significant (p={p_str}), reject H₀"
        sig = True
    elif p_val < 0.10:
        conclusion = f" Marginally significant (p={p_str}), cannot reject H₀"
        sig = False
    else:
        conclusion = f" Not significant (p={p_str}), cannot reject H₀"
        sig = False
    print(f"  {conclusion}")

    # ── Step 8: Wilcoxon signed-rank test ──
    wilcoxon_sig = None
    if norm_ok is False:
        nonzero = diffs[diffs != 0]
        if len(nonzero) >= 5:
            w_stat, w_p = stats.wilcoxon(nonzero, alternative='greater')
            w_p_str = f"{w_p:.2e}" if w_p < 1e-6 else f"{w_p:.6f}"
            wilcoxon_sig = w_p < alpha
            print(f"  Wilcoxon fallback: W={w_stat:.1f}, p={w_p_str}"
                  f" → {' Significant' if wilcoxon_sig else ' Not significant'}")

    # Collect for summary table
    summary.append({
        'metric': metric_name, 'mean_diff': mean_d, 't': t_stat,
        'df': df, 'p': p_val, 'd': cohens_d, 'norm': norm_ok,
        'sig': sig, 'wilcoxon': wilcoxon_sig,
    })

# ── Summary table across all metrics ──
print(f"\n{'=' * 70}")
print(f"  SUMMARY TABLE")
print(f"{'=' * 70}")
print(f"  {'Metric':<24} {'MeanDiff':>10} {'t':>8} {'p-value':>12} {'Cohen d':>9} {'Norm':>5} {'Sig':>5}")
print(f"  {'─' * 75}")
for s in summary:
    p_str = f"{s['p']:.2e}" if s['p'] < 1e-6 else f"{s['p']:.6f}"
    n_str = "T" if s['norm'] else ("WARN" if s['norm'] is False else "—")
    sig_str = "T" if s['sig'] else "F"
    print(f"  {s['metric']:<24} {s['mean_diff']:>+10.4f} {s['t']:>8.3f} {p_str:>12} {s['d']:>9.4f} {n_str:>5} {sig_str:>5}")

print(f"\n  Cohen's d interpretation: |d|<0.2 negligible, ≈0.2 small, ≈0.5 medium, ≈0.8 large, >1.0 very large")
print(f"{'=' * 70}")

# **PSNR movement**

In [ ]:
# ============================================================
# Per-Step Latent PSNR: DeepCache vs Linear Schedule
# ============================================================

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ────────────────────────────────────────────────────────────
# 1. Latent PSNR helper
# ────────────────────────────────────────────────────────────
def compute_latent_psnr(lat_ref, lat_gen):
    #PSNR between two latent tensors (same shape)
    a = lat_ref.float().cpu()
    b = lat_gen.float().cpu()
    mse = torch.mean((a - b) ** 2).item()
    if mse == 0:
        return 100.0
    max_val = torch.max(torch.abs(a)).item()
    return float(20 * np.log10(max_val / (np.sqrt(mse) + 1e-8)))

# ────────────────────────────────────────────────────────────
# 2. Trajectory capture via scheduler.step monkey-patch
# ────────────────────────────────────────────────────────────
class LatentTrajectoryCapture:

    def __init__(self, scheduler):
        self.scheduler = scheduler
        self.trajectory = []
        self._original_step = None

    def enable(self):
        self.trajectory = []
        self._original_step = self.scheduler.step

        capture = self  # closure reference

        def patched_step(*args, **kwargs):
            result = capture._original_step(*args, **kwargs)
            # result[0] is the prev_sample (denoised latent at this step)
            if isinstance(result, tuple):
                latent = result[0]
            else:
                latent = result.prev_sample
            capture.trajectory.append(latent.detach().clone().cpu())
            return result

        self.scheduler.step = patched_step

    def disable(self):
        if self._original_step is not None:
            self.scheduler.step = self._original_step
            self._original_step = None

    def get_trajectory(self):
        return list(self.trajectory)

# ────────────────────────────────────────────────────────────
# 3. Generation functions with trajectory capture
# ────────────────────────────────────────────────────────────
def gen_baseline_traj(prompt, steps=50, seed=42):
    set_seed(seed)
    cap = LatentTrajectoryCapture(pipe.scheduler)
    cap.enable()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    cap.disable()
    return img, cap.get_trajectory()


def gen_deepcache_traj(prompt, steps=50, seed=42,
                       cache_interval=3, cache_branch_id=0):
    from DeepCache import DeepCacheSDHelper
    set_seed(seed)
    h = DeepCacheSDHelper(pipe=pipe)
    h.set_params(cache_interval=cache_interval, cache_branch_id=cache_branch_id)
    h.enable()

    cap = LatentTrajectoryCapture(pipe.scheduler)
    cap.enable()
    with torch.inference_mode():
        img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    cap.disable()
    h.disable()
    return img, cap.get_trajectory()


def gen_linear_traj(prompt, steps, seed, full_step_list, branch_id=0):
    set_seed(seed)
    cfg = ScheduleCacheConfig(total_steps=steps, cache_branch_id=branch_id)
    helper = ScheduleCacheHelper(pipe, cfg, full_step_list)

    cap = LatentTrajectoryCapture(pipe.scheduler)
    cap.enable()
    with helper.enabled():
        with torch.inference_mode():
            img = pipe(prompt, num_inference_steps=steps, output_type="pil").images[0]
    cap.disable()
    return img, cap.get_trajectory()


# ────────────────────────────────────────────────────────────
# 4. Run per-step PSNR experiment
# ────────────────────────────────────────────────────────────
TRAJ_SAVE_DIR = os.path.join(SAVE_DIR, "trajectory")
os.makedirs(TRAJ_SAVE_DIR, exist_ok=True)

# CSV for per-step PSNR data
traj_csv_path = os.path.join(TRAJ_SAVE_DIR, "per_step_psnr.csv")
traj_csv_fields = ["prompt_idx", "prompt", "step", "psnr_dc", "psnr_linear"]

with open(traj_csv_path, "w", newline="", encoding="utf-8") as tf:
    tw = csv.DictWriter(tf, fieldnames=traj_csv_fields)
    tw.writeheader()

    # Collect all per-step PSNR curves for averaging later
    all_psnr_dc = []
    all_psnr_li = []

    for pi, prompt in enumerate(prompts):
        print(f"\n{'='*60}")
        print(f"  Trajectory {pi+1}/{len(prompts)}")
        print(f"  Prompt: \"{prompt[:80]}{'...' if len(prompt)>80 else ''}\"")
        print(f"{'='*60}")

        # 1) Baseline trajectory
        print("  [1/3] Baseline trajectory ...")
        img_bl, traj_bl = gen_baseline_traj(prompt, NUM_STEPS, SEED)
        print(f"        {len(traj_bl)} steps captured")

        # 2) DeepCache trajectory
        print(f"  [2/3] DeepCache trajectory (interval={DC_INTERVAL}) ...")
        img_dc, traj_dc = gen_deepcache_traj(
            prompt, NUM_STEPS, SEED, DC_INTERVAL, DC_BRANCH)
        print(f"        {len(traj_dc)} steps captured")

        # 3) Linear Schedule trajectory
        print(f"  [3/3] Linear Schedule trajectory (K={FULL_BUDGET}) ...")
        img_li, traj_li = gen_linear_traj(
            prompt, NUM_STEPS, SEED, linear_schedule, branch_id=DC_BRANCH)
        print(f"        {len(traj_li)} steps captured")

        # Compute per-step PSNR
        n_steps = min(len(traj_bl), len(traj_dc), len(traj_li))
        psnr_dc_curve = []
        psnr_li_curve = []

        for s in range(n_steps):
            p_dc = compute_latent_psnr(traj_bl[s], traj_dc[s])
            p_li = compute_latent_psnr(traj_bl[s], traj_li[s])
            psnr_dc_curve.append(p_dc)
            psnr_li_curve.append(p_li)

            tw.writerow({
                "prompt_idx": pi + 1,
                "prompt": prompt[:100],
                "step": s,
                "psnr_dc": f"{p_dc:.2f}",
                "psnr_linear": f"{p_li:.2f}",
            })

        tf.flush()
        all_psnr_dc.append(psnr_dc_curve)
        all_psnr_li.append(psnr_li_curve)

        # ── Per-prompt plot ──
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(range(n_steps), psnr_dc_curve,
                label=f"DeepCache (interval={DC_INTERVAL})",
                marker='.', markersize=3, linewidth=1.2, color='tab:blue')
        ax.plot(range(n_steps), psnr_li_curve,
                label=f"Linear Schedule (K={FULL_BUDGET})",
                marker='.', markersize=3, linewidth=1.2, color='tab:orange')

        # Mark full steps on x-axis
        dc_positions = list(range(0, NUM_STEPS, DC_INTERVAL))
        for fs in dc_positions:
            if fs < n_steps:
                ax.axvline(x=fs, color='tab:blue', alpha=0.15, linewidth=0.8)
        for fs in linear_schedule:
            if fs < n_steps:
                ax.axvline(x=fs, color='tab:orange', alpha=0.15, linewidth=0.8)

        ax.set_xlabel("Timestep Index")
        ax.set_ylabel("Latent PSNR (dB) vs Baseline")
        ax.set_title(f"Prompt {pi+1}: \"{prompt[:60]}{'...' if len(prompt)>60 else ''}\"")
        ax.legend(loc='lower left')
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        fig_path = os.path.join(TRAJ_SAVE_DIR, f"psnr_prompt_{pi+1}.png")
        fig.savefig(fig_path)
        print(f"  Saved -> {fig_path}")
        try:
            from IPython.display import display
            display(fig)
        except:
            pass
        plt.close(fig)

        # Free memory
        del traj_bl, traj_dc, traj_li
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ──────────────────────────────────────────────────────────
# 5. Average PSNR curve across all prompts
# ──────────────────────────────────────────────────────────
if len(all_psnr_dc) > 0:
    min_len = min(len(c) for c in all_psnr_dc)
    avg_dc = np.mean([c[:min_len] for c in all_psnr_dc], axis=0)
    avg_li = np.mean([c[:min_len] for c in all_psnr_li], axis=0)
    std_dc = np.std([c[:min_len] for c in all_psnr_dc], axis=0)
    std_li = np.std([c[:min_len] for c in all_psnr_li], axis=0)

    fig2, ax2 = plt.subplots(figsize=(12, 5))

    ax2.plot(range(min_len), avg_dc,
             label=f"DeepCache (interval={DC_INTERVAL})",
             linewidth=2, color='tab:blue')
    ax2.fill_between(range(min_len), avg_dc - std_dc, avg_dc + std_dc,
                     alpha=0.15, color='tab:blue')

    ax2.plot(range(min_len), avg_li,
             label=f"Linear Schedule (K={FULL_BUDGET})",
             linewidth=2, color='tab:orange')
    ax2.fill_between(range(min_len), avg_li - std_li, avg_li + std_li,
                     alpha=0.15, color='tab:orange')

    # Mark full step positions
    dc_positions = list(range(0, NUM_STEPS, DC_INTERVAL))
    for fs in dc_positions:
        if fs < min_len:
            ax2.axvline(x=fs, color='tab:blue', alpha=0.12, linewidth=0.8)
    for fs in linear_schedule:
        if fs < min_len:
            ax2.axvline(x=fs, color='tab:orange', alpha=0.12, linewidth=0.8)

    ax2.set_xlabel("Timestep Index")
    ax2.set_ylabel("Latent PSNR (dB) vs Baseline")
    ax2.set_title(
        f"Average Per-Step PSNR ({len(all_psnr_dc)} prompts)  |  "
        f"K={FULL_BUDGET}, branch={DC_BRANCH}")
    ax2.legend(loc='lower left')
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()

    avg_path = os.path.join(TRAJ_SAVE_DIR, "psnr_average.png")
    fig2.savefig(avg_path)
    print(f"\nAverage PSNR plot -> {avg_path}")
    try:
        from IPython.display import display
        display(fig2)
    except:
        pass
    plt.close(fig2)

    # Print summary table
    print(f"\n{'='*60}")
    print(f"  PER-STEP PSNR SUMMARY ({len(all_psnr_dc)} prompts)")
    print(f"{'='*60}")
    print(f"  {'':>10} {'DeepCache':>12} {'Linear':>12}")
    print(f"  {'─'*36}")
    print(f"  {'Mean PSNR':>10} {np.mean(avg_dc):>12.2f} {np.mean(avg_li):>12.2f}")
    print(f"  {'Min PSNR':>10}  {np.min(avg_dc):>12.2f} {np.min(avg_li):>12.2f}")
    print(f"  {'Last step':>10} {avg_dc[-1]:>12.2f} {avg_li[-1]:>12.2f}")
    print(f"  {'─'*36}")

    # Find where Linear beats DeepCache and vice versa
    li_better = np.sum(avg_li > avg_dc)
    dc_better = np.sum(avg_dc > avg_li)
    print(f"  Linear > DeepCache in {li_better}/{min_len} steps")
    print(f"  DeepCache > Linear in {dc_better}/{min_len} steps")

print(f"\n  CSV -> {traj_csv_path}")
print("Done!")